# 🎓 Education Regulation Impact Analyzer (ERIA) - Core Engine Demo

Welcome to the **ERIA** interactive Jupyter Notebook! This notebook contains the step-by-step implementation of the core Natural Language Processing (NLP) and Artificial Intelligence (AI) logic used to analyze educational regulations and academic circulars.

### 🎯 Key Objectives Demonstrated:
1. **Document Ingestion:** Reading standard PDF circulars and scraping official notice links (UGC/AICTE).
2. **AI Analytics:** Connecting to the **Google Gemini API** (`gemini-1.5-flash`) to perform classification, layman summarization, stakeholder impact mapping, chronology building, and impact forecasting.
3. **Structured Outputs:** Parsing AI results into neat Python lists, dictionaries, and Pandas DataFrames for quick academic profiling.

--- 
## 🛠️ Step 1: Install & Import Dependencies

In [ ]:
# Install required packages if not already present
!pip install google-generativeai pypdf beautifulsoup4 requests pandas

In [ ]:
import os
import re
import json
import requests
from bs4 import BeautifulSoup
import pypdf
import pandas as pd
import google.generativeai as genai

--- 
## 📂 Step 2: Document Ingestion and Web Scraping Engine

Here we define utility functions to read local PDF circulars and scrape official notifications from the web.

In [ ]:
def extract_text_from_pdf(pdf_path):
    """
    Extracts plain text from a local PDF circular.
    """
    try:
        reader = pypdf.PdfReader(pdf_path)
        text_content = []
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text:
                text_content.append(text)
        return "\n".join(text_content)
    except Exception as e:
        return f"Error reading PDF: {e}"

def scrape_url_content(url):
    """
    Scrapes official notice boards (e.g. UGC, AICTE) and cleans boilerplate.
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Strip styling, script, navigation, footer
        for elem in soup(["script", "style", "nav", "footer", "header", "aside"]):
            elem.decompose()
            
        lines = []
        for element in soup.find_all(['h1', 'h2', 'h3', 'p', 'li']):
            text = element.get_text().strip()
            if text:
                lines.append(text)
                
        cleaned_text = "\n\n".join(lines)
        return re.sub(r'\n{3,}', '\n\n', cleaned_text)
    except Exception as e:
        return f"Error scraping URL: {e}"

--- 
## 🤖 Step 3: Define the Gemini API Core Analytics Engine

We write the structured NLP analyzer that prompts the Gemini model to parse the regulation details and return a strictly validated JSON structure.

In [ ]:
def analyze_regulation(text, api_key):
    """
    Utilizes the Google Gemini API to analyze the regulation text and extract key compliance insights.
    """
    # 1. Configure the Gemini API
    genai.configure(api_key=api_key)
    
    # 2. Build structured prompt instructing the model to yield a perfect JSON object
    prompt = f"""
    You are an expert Education Policy Analyst and Compliance Officer.
    Dissect the following official university/governmental circular text and convert it into a structured summary report.
    
    Document Text:
    \"\"\"
    {text}
    \"\"\"
    
    Return your response strictly in JSON format. Do not include markdown code block wrappers (like ```json). The output must contain:
    - "document_title"
    - "category" (Accreditation, Scholarship, Curriculum, Faculty Policy, Examination, etc.)
    - "sentiment" (High Compliance Workload, Flexible & Favorable, Operationally Disruptive, etc.)
    - "layman_summary" (List of 5-10 key points in conversational words)
    - "stakeholders" (Dict mapping "students", "faculty", "institutions", "compliance" to dictionaries containing "positives", "negatives", and "opportunities")
    - "chronology" (List of dictionaries mapping past related policies containing: "event", "date", "context")
    - "impact_forecast" (Dict containing lists for: "short_term", "medium_term", "long_term", and "risks_and_readiness")
    """
    
    # 3. Request analysis using the high-performance gemini-1.5-flash model
    model = genai.GenerativeModel('gemini-1.5-flash')
    generation_config = {
        "temperature": 0.2,
        "response_mime_type": "application/json"
    }
    
    response = model.generate_content(prompt, generation_config=generation_config)
    raw_json_str = response.text.strip()
    
    # Parse the output into a Python Dict
    return json.loads(raw_json_str)

--- 
## 🧪 Step 4: Run a Demo Test Case

We load a sample UGC Circular on the **Academic Bank of Credits (ABC)** (NEP operationalization policy) and run our analysis.

In [ ]:
sample_ugc_credits_circular = """
UNIVERSITY GRANTS COMMISSION (UGC)
REGULATIONS, 2026

No. F. 1-1/2026(UGC/ABC) — In exercise of the powers conferred by the UGC Act, 1956, the UGC hereby operationalizes the digital Academic Bank of Credits (ABC) to facilitate student mobility and multiple exit/entry pathways across all Indian higher education institutions.

Key Rules:
1. Mandate: All Central, State, and accredited private colleges must register on the national ABC portal within six months.
2. Digital Credit Logs: Student course credits must be logged under their unique ABC ID. Credits remain valid for seven (7) academic years.
3. Exit/Entry: Students can exit after Year 1 with a Certificate (40 credits), Year 2 with a Diploma (80 credits), or Year 3 with a Bachelor's Degree (120 credits).
4. Sanctions: HEIs failing to register on the platform shall face warnings during NAAC evaluations and suspension of UGC developmental grants.
"""

# ⚠️ Please configure your API key below to run this cell!
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_API_KEY_HERE")

if GEMINI_API_KEY == "YOUR_API_KEY_HERE":
    print("❌ Please replace 'YOUR_API_KEY_HERE' with your actual Google Gemini API key to run this cell.")
else:
    print("🚀 Starting ERIA analysis...")
    analysis_output = analyze_regulation(sample_ugc_credits_circular, GEMINI_API_KEY)
    print("✅ Analysis Successful! Output saved in 'analysis_output'.")

--- 
## 📊 Step 5: Structuring & Displaying the AI Analysis Results

Let's look at how we can cleanly isolate and display the results in Pandas tables.

In [ ]:
# If cell above failed, load mock outputs so this cell always demonstrates formatting
try:
    data = analysis_output
except NameError:
    data = {
        "document_title": "UGC Academic Bank of Credits (ABC) Regulations, 2026",
        "category": "Accreditation & Curriculum",
        "sentiment": "High Compliance Workload",
        "layman_summary": [
            "Establishment of a national digital credit vault (ABC) for students.",
            "Mandatory university registration on the credit platform within 6 months.",
            "Credits remain valid for 7 years before expiring.",
            "Enables students to leave with certificates or diplomas and re-enter later."
        ],
        "stakeholders": {
            "students": {
                "positives": ["Credit transfer across colleges", "Flexible exit options"],
                "negatives": ["7-year shelf life limit on credits"],
                "opportunities": ["Seamless multi-university learning"]
            },
            "faculty": {
                "positives": ["Standardized grading guidelines"],
                "negatives": ["Added administrative grading paperwork"],
                "opportunities": ["Developing credit-based short courses"]
            }
        }
    }

print(f"📌 Document Title: {data['document_title']}")
print(f"🏷️ Classification: {data['category']}")
print(f"⚖️ Tone Index:    {data['sentiment']}\n")

print("🧠 Layman Summary Bullet Points:")
for item in data['layman_summary']:
    print(f"  • {item}")

In [ ]:
# Display Stakeholder Positives & Negatives in a neat DataFrame
rows = []
for stakeholder, metrics in data.get('stakeholders', {}).items():
    pos = ", ".join(metrics.get('positives', []))
    neg = ", ".join(metrics.get('negatives', []))
    opp = ", ".join(metrics.get('opportunities', []))
    rows.append({
        "Stakeholder Role": stakeholder.capitalize(),
        "Positives/Benefits": pos,
        "Constraints/Negatives": neg,
        "Future Avenues": opp
    })

df_stakeholders = pd.DataFrame(rows)
df_stakeholders